# GraphSAGE Node Classification via Aura Session (typed endpoints)

Train a supervised GraphSAGE model on Cora and run node classification inference, using the typed
`gds.graph_sage.supervised` endpoints introduced in graphdatascience-client PR #1289
(instead of raw Arrow `JobClient` calls).

In [ ]:
%pip install "graphdatascience>=2.0a1" python-dotenv

## Configuration

In [ ]:
SESSION_NAME = "graphsage-nc-via-session-typed"
CLOUD_PROVIDER, CLOUD_REGION = "gcp", "us-east4"

## Credentials

In [ ]:
import os

from dotenv import load_dotenv

# This allows to load required secrets from `.env` file in local directory
load_dotenv(".env")

## Get or Create Session

In [ ]:
from datetime import timedelta

from graphdatascience.session import AuraAPICredentials, CloudLocation, GdsSessions, SessionMemory

api_credentials = AuraAPICredentials(
    client_id=os.environ["CLIENT_ID"],
    client_secret=os.environ["CLIENT_SECRET"],
    project_id=os.environ["PROJECT_ID"],
)

sessions = GdsSessions(api_credentials=api_credentials)

gds = sessions.get_or_create(
    session_name=SESSION_NAME,
    memory=SessionMemory.m_4GB,
    ttl=timedelta(minutes=30),
    gpu=True,
    cloud_location=CloudLocation(CLOUD_PROVIDER, CLOUD_REGION),
)

gds.verify_connectivity()

## Build Catalog Arrow Endpoints

In [ ]:
from graphdatascience.procedure_surface.arrow.catalog.catalog_arrow_endpoints import CatalogArrowEndpoints

arrow_client = gds._authenticated_arrow_client
catalog_arrow_endpoints = CatalogArrowEndpoints(arrow_client)

## Load Cora Dataset

Cora `Paper` nodes carry a numeric `features` vector and a `subject` class label.
The `subject` property is used as the supervised training target.

In [ ]:
_ = catalog_arrow_endpoints.drop("cora", fail_if_missing=False)
G = catalog_arrow_endpoints.datasets.load_cora()
print("Loaded graph:", G.name())

## Train GraphSAGE (Supervised)

`gds.graph_sage.supervised.train` submits the training job, blocks until completion (with progress
reporting), and returns the trained model together with a train summary result.

In [ ]:
modelname = "gs_cora_nc_typed"

In [ ]:
model, train_result = gds.graph_sage.supervised.train(
    G,
    model_name=modelname,
    feature_properties=["subject", "features"],
    target_label="Paper",
    target_property="subject",
    epochs=15,
    epochs_per_val=2,
    embedding_dimension=256,
    learning_rate=0.001,
    num_neighbors=[20, 20],
    batch_size=512,
    activation_function="relu",
    aggregator="mean",
    random_seed=42,
)

print("Trained model:", model.name())

In [ ]:
train_result

## Run Inference

`model.predict_stream` runs the predict job against the stored model and streams the results.
Supervised GraphSAGE returns one row per node with `predictedClass` and `predictedProbabilities`.

In [ ]:
predictions_df = model.predict_stream(
    G,
    feature_properties=["subject", "features"],
    random_seed=42,
)

assert set(predictions_df.columns) == {"nodeId", "predictedClass", "predictedProbabilities"}
predictions_df

## Results

In [ ]:
print(f"Predicted classes for {len(predictions_df)} nodes")
predictions_df["predictedClass"].value_counts()

## (Optional) cleanup

In [ ]:
gds.model.delete(modelname, fail_if_missing=False)
sessions.delete(session_name=SESSION_NAME)